### Understanding Domain Adaption and Handling Domain Specific Data
* Why domain adaption?
    * Many Real-world tasks involve domain-specific data
    * Pre-trained models on general datasets might not perform optimally without adaption
* Steps in domain adaption
    * Fine-Tune the pre-trained model on a domain-specific dataset
    * Incorporate domain-specific embeddings or vocabulary
    * Apply additional pre-training on the domain data if necessary
---
### Challenges

* Data Mismatch
    * The souce domain may not represent the target domain adequateky
    * Eg: General Text (Wikipedia) vs. technical medical jargon
* Catasthropic Forgetting
    * During fine-tuning , the model may forget the knowledge learned fro, the source domain
* Computational Constraints
    * Fine-Tuning large pre-trained models (eg: BERT, GPT), requires significant computational resources

==============================================================================================================

### Strategies to address Challenges
* Transfer learning from related domains
    * Fine tuning on an intermediate domain dataset before adpating to the target domain
* Data Augmentation
    * Generate Synthetic domain-specific data to augment the target dataset
    * Eg:
        * Use paraphrasing techniques to increase dataset diversity
* Domain-Sepcific Embeddings
    * Use pre-trained embeddings tailored to the target domain (eg: BioBERT for biomedical data, LegalBERT for legal Text) 
    

In [1]:
!pip install transformers[torch]
from datasets import load_dataset
from transformers import AutoTokenizer

In [2]:
dataset = load_dataset("armanc/pubmed-rct20k")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Repo card metadata block was not found. Setting CardData to empty.


In [10]:
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")

def preprocess_data(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

In [22]:
tokenized_dataset = dataset.map(preprocess_data, batched=True)
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
label_names = tokenized_dataset["train"].unique("labels")
label2id = {label: idx for idx, label in enumerate(sorted(label_names))}

def encode_labels(example):
    example["labels"] = label2id[example["labels"]]
    return example

tokenized_dataset = tokenized_dataset.map(encode_labels)


tokenized_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "labels"]
)

Map:   0%|          | 0/176642 [00:00<?, ? examples/s]

Map:   0%|          | 0/29672 [00:00<?, ? examples/s]

Map:   0%|          | 0/29578 [00:00<?, ? examples/s]

Map:   0%|          | 0/176642 [00:00<?, ? examples/s]

Map:   0%|          | 0/29672 [00:00<?, ? examples/s]

Map:   0%|          | 0/29578 [00:00<?, ? examples/s]

In [23]:
from transformers import AutoModelForSequenceClassification
id2label = {idx: label for label, idx in label2id.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those

In [24]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    report_to="none",
    save_total_limit=2
)

In [7]:
from transformers import Trainer

In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer
)

In [19]:
!pip install -U datasets
import sys
sys.modules.pop("torchvision", None)
sys.modules.pop("torchvision.io", None)

In [ ]:

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.373640,0.352265
2,0.311966,0.355161


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
trainer.save_model("./biobert-pubmed-rct")
tokenizer.save_pretrained("./biobert-pubmed-rct")
print("model saved locally")

In [ ]:

from huggingface_hub import login, HfApi

token = "<token>"
login(token=token)

repo_id = "Achiket/biobert-pubmed-rct-classification"


 
api = HfApi()
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(
    folder_path="./biobert-pubmed-rct",
    repo_id=repo_id,
)

print("pushed to https://huggingface.co/" + repo_id)

In [ ]:

# Evaluate model
results = trainer.evaluate()
print("Evaluation Results:",results)

In [ ]:
import random

def augment_text(text):
    synonyms = {"cancer": ["tumor", "malignancy"], "study": ["research", "experiment"]}
    words = text.split()
    new_words = [random.choice(synonyms[word]) if word in synonyms else word for word in words]
    return " ".join(new_words)

In [ ]:
augmented_data = [augment_text(sample["text"]) for sample in dataset["train"]]
print(augmented_data[:5])